<a href="https://colab.research.google.com/github/plmanognya32/formulaonestorytelling/blob/main/notebooks/02_load_to_snowflake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

!pip install snowflake-connector-python --quiet
print("Ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready.


In [36]:
import snowflake.connector
import pandas as pd
import json
import os
from google.colab import userdata

SNOWFLAKE_ACCOUNT = "QMDFLLI-CS77896"
SNOWFLAKE_USER = "PLMANOGNYA32"
SNOWFLAKE_PASSWORD = userdata.get('SNOWFLAKE_PASSWORD')
SNOWFLAKE_WAREHOUSE = "COMPUTE_WH"
SNOWFLAKE_DATABASE = "formulaoneengine"

conn = snowflake.connector.connect(
    account = SNOWFLAKE_ACCOUNT,
    user = SNOWFLAKE_USER,
    password = SNOWFLAKE_PASSWORD,
    warehouse = SNOWFLAKE_WAREHOUSE,
    database = SNOWFLAKE_DATABASE,
    schema = "RAW"
)

cursor = conn.cursor()
cursor.execute("SELECT CURRENT_VERSION()")
print("Connected. Snowflake version: ", cursor.fetchone()[0])

Connected. Snowflake version:  10.14.103


In [18]:
def load_df_to_snowflake(df, table_name, cursor):
  df = df.astype(object).where(pd.notnull(df), None)

  cols = list(df.columns)
  placeholders = ', '.join(['%s'] * len(cols))
  col_names = ', '.join(cols)

  sql = f"INSERT INTO {table_name} ({col_names}) VALUES ({placeholders})"

  rows = [tuple(row) for row in df.itertuples(index=False)]

  chunk_size = 1000
  total = 0
  for i in range(0, len(rows), chunk_size):
    chunk = rows[i:i+chunk_size]
    cursor.executemany(sql, chunk)
    total += len(chunk)
    print(f"Loaded {total}/{len(rows)} rows into {table_name}")

  print(f"{table_name}: {total} rows loaded\n")

DATA_PATH = '/content/drive/MyDrive/f1_engine/data'
print("Function ready. Data path:", DATA_PATH)


Function ready. Data path: /content/drive/MyDrive/f1_engine/data


In [21]:
laps_df = pd.read_json(f'{DATA_PATH}/monaco_2023_laps.json')

laps_load = laps_df[[
    'Driver', 'LapNumber', 'Position', 'LapTime', 'Compound', 'TyreLife', 'Stint', 'Time', 'PitInTime', 'PitOutTime', 'IsPersonalBest', 'Team'
]].copy()

laps_load.columns = [
    'driver', 'lap_number', 'race_position', 'lap_time', 'compound', 'tyre_life', 'stint', 'cumulative_time', 'pit_in_time', 'pit_out_time', 'is_personal_best', 'team'
]

for col in ['lap_time', 'cumulative_time', 'pit_in_time', 'pit_out_time']:
  laps_load[col] = laps_load[col].astype(str).replace('nan', None)

print(f"Preparing {len(laps_load)} lap rows...")
load_df_to_snowflake(laps_load, 'LAPS', cursor)

Preparing 1515 lap rows...
Loaded 1000/1515 rows into LAPS
Loaded 1515/1515 rows into LAPS
LAPS: 1515 rows loaded



In [24]:
telem_df = pd.read_json(f'{DATA_PATH}/monaco_2023_ver_telemetry.json')

telem_load = telem_df[['Distance','Speed','RPM','Throttle','Brake','DRS','nGear','Time']].copy()
telem_load.columns = ['distance','speed','rpm','throttle','brake','drs','gear','time']

telem_load['driver'] = 'VER'
telem_load['session_key'] = 'monaco_2023_R'
telem_load['time'] = telem_load['time'].astype(str)

print(f"Preparing to load {len(telem_load)} telemetry rows...")
load_df_to_snowflake(telem_load, 'TELEMETRY', cursor)

Preparing to load 578 telemetry rows...
Loaded 578/578 rows into TELEMETRY
TELEMETRY: 578 rows loaded



In [25]:
# Pit stops
pits_df = pd.read_json(f'{DATA_PATH}/monaco_2023_pit_stops.json')
pits_load = pits_df[['Driver','LapNumber','Stint','Compound','TyreLife','PitInTime','PitOutTime','pit_duration']].copy()
pits_load.columns = ['driver','lap_number','stint','compound','tyre_life','pit_in_time','pit_out_time','pit_duration']
for col in ['pit_in_time','pit_out_time','pit_duration']:
    pits_load[col] = pits_load[col].astype(str).replace('nan', None)

print(f"Loading {len(pits_load)} pit stop rows...")
load_df_to_snowflake(pits_load, 'PIT_STOPS', cursor)

# Weather
weather_df = pd.read_json(f'{DATA_PATH}/monaco_2023_weather.json')
weather_load = weather_df[['Time','AirTemp','TrackTemp','Humidity','Rainfall','WindSpeed','WindDirection']].copy()
weather_load.columns = ['time','air_temp','track_temp','humidity','rainfall','wind_speed','wind_direction']
weather_load['time'] = weather_load['time'].astype(str)

print(f"Loading {len(weather_load)} weather rows...")
load_df_to_snowflake(weather_load, 'WEATHER', cursor)

Loading 38 pit stop rows...
Loaded 38/38 rows into PIT_STOPS
PIT_STOPS: 38 rows loaded

Loading 176 weather rows...
Loaded 176/176 rows into WEATHER
WEATHER: 176 rows loaded



In [37]:
tables = ['LAPS', 'TELEMETRY', 'PIT_STOPS', 'WEATHER']

print("Snowflake Row Counts")
for table in tables:
    cursor.execute(f"SELECT COUNT(*) FROM RAW.{table}")
    count = cursor.fetchone()[0]
    status = "Okay" if count > 0 else "Empty"
    print(f"  {status}  RAW.{table}: {count:,} rows")

print("\nTiming Tower Test (lap 30)")
cursor.execute("""
    SELECT driver, race_position, compound, tyre_life
    FROM ANALYTICS.TIMING_TOWER
    WHERE lap_number = 30
    ORDER BY race_position
    LIMIT 5
""")
for row in cursor.fetchall():
    print(f"  P{row[1]} | {row[0]} | {row[2]} | {row[3]} laps old")

cursor.close()
conn.close()
print("\nDone. Connection closed.")

Snowflake Row Counts
  Okay  RAW.LAPS: 1,515 rows
  Okay  RAW.TELEMETRY: 1,156 rows
  Okay  RAW.PIT_STOPS: 76 rows
  Okay  RAW.WEATHER: 352 rows

Timing Tower Test (lap 30)
  P1 | VER | MEDIUM | 30 laps old
  P2 | ALO | HARD | 31 laps old
  P3 | OCO | MEDIUM | 30 laps old
  P4 | SAI | HARD | 30 laps old
  P5 | HAM | MEDIUM | 30 laps old

Done. Connection closed.
